# Vulnerability Management RAG — Colab Simulation

Test-drive the multi-agent RAG vulnerability bot from the [Portfolio repo](https://github.com/Konaguy/Portfolio/tree/claude/ai-secops-rag) end to end, in the browser.

**What this notebook does**
1. Clones the repo and installs the core dependencies.
2. Ingests the bundled sample Tenable export + security policies into an in-memory Qdrant vector store.
3. Runs the LangGraph agent pipeline (router → retriever → specialist) and lets you ask it questions.
4. Simulates an **Azure Defender / MDVM scan** of 2 servers + 2 Windows 11 PCs and asks the bot about the findings.

It runs **fully offline by default** using the deterministic stub LLM and a dependency-free embedder — no API key, no services. Add an Anthropic API key in the optional cell for real Claude-authored answers.

> Runtime → *Run all*, or run the cells top to bottom.


## 1 · Clone the repo and install dependencies

In [3]:
!git clone --depth 1 --branch claude/ai-secops-rag https://github.com/Konaguy/Portfolio.git 2>/dev/null || echo "already cloned"
%cd Portfolio/vuln-management-rag

# Core deps only. (Chainlit + sentence-transformers are NOT needed for the
# headless simulation; the optional UI cell at the end installs Chainlit.)
!pip install -q "anthropic>=0.40" "langgraph>=0.2.28" "qdrant-client>=1.9" "pydantic>=2.5"
print("\nInstalled. Working dir:")
!pwd


/content/Portfolio/vuln-management-rag/Portfolio/vuln-management-rag

Installed. Working dir:
/content/Portfolio/vuln-management-rag/Portfolio/vuln-management-rag


## 2 · (Optional) Anthropic API key

Leave the prompt **blank** to run offline with the deterministic stub LLM (great for seeing the pipeline wiring). Paste a key to get real Claude-authored, grounded answers.

In [4]:
import os, getpass
key = getpass.getpass("Anthropic API key (optional — press Enter to skip): ").strip()
if key:
    os.environ["ANTHROPIC_API_KEY"] = key
print("LLM:", "Claude (" + os.environ.get("ANTHROPIC_MODEL", "claude-opus-5") + ")" if os.environ.get("ANTHROPIC_API_KEY") else "offline stub (no key)")


KeyboardInterrupt: Interrupted by user

## 3 · Build the corpus and the agent graph

Ingests the sample Tenable findings and the two security policies into an in-memory Qdrant collection, then wires up the LangGraph graph.

In [5]:
import os
from vulnrag.config import Settings
from vulnrag.embeddings import build_embedder
from vulnrag.vectorstore import VectorStore
from vulnrag.ingest import parse_tenable_file, parse_policy_dir
from vulnrag.agents.graph import VulnRagGraph
from vulnrag.llm import ClaudeClient, FakeLLMClient

settings = Settings.from_env()   # defaults: in-memory Qdrant, dependency-free hashing embedder
store = VectorStore(settings, build_embedder(settings))
n_v = store.upsert_vulnerabilities(parse_tenable_file("data/sample_tenable_export.csv"))
n_p = store.upsert_policies(parse_policy_dir("data/sample_policies"))

llm = ClaudeClient(model=settings.anthropic_model) if os.environ.get("ANTHROPIC_API_KEY") else FakeLLMClient()
graph = VulnRagGraph.from_settings(settings, llm, store=store)

print(f"Ingested {n_v} vulnerability findings + {n_p} policy chunks  ->  {store.count()} corpus chunks")


Ingested 10 vulnerability findings + 9 policy chunks  ->  19 corpus chunks


## 4 · Ask the bot

The helper prints the structured answer: summary, ordered remediation steps, the prioritization rationale, the **citations** (chunk IDs it grounded on — a hallucinated citation is rejected upstream), and the agent **trace** (which intent it routed to and how many chunks it retrieved).

In [6]:
def ask(q):
    final = graph.answer(q)
    a = final["answer"]
    print("Q:", q)
    print("\nSummary:\n ", a["summary"])
    if a["remediation_steps"]:
        print("\nRemediation steps:")
        for i, s in enumerate(a["remediation_steps"], 1):
            print(f"  {i}. {s}")
    if a["prioritization_rationale"]:
        print("\nPrioritization:", a["prioritization_rationale"])
    print("\nCitations:", ", ".join(a["cited_chunk_ids"]) or "(none)")
    print("Trace:   ", " -> ".join(final.get("trace", [])))
    print("=" * 78)

ask("How do I remediate the Log4Shell findings on the DMZ hosts?")
ask("Which of the critical vulnerabilities should I patch first?")
ask("What is our SLA for patching critical vulnerabilities?")


Q: How do I remediate the Log4Shell findings on the DMZ hosts?

Summary:
  Based on the retrieved findings and policy context, prioritize the critical/high items on internet-facing hosts and patch within the policy SLA.

Remediation steps:
  1. Apply the vendor-listed patch or upgrade for the affected plugin.
  2. Re-scan the host to confirm the finding is resolved.
  3. Record the change and closure in the ticketing system.

Prioritization: Ranked by severity and VPR, weighted toward exposed hosts, against the remediation SLA in the retrieved policy.

Citations: pol-bdf56af187ef, pol-cb8cb70cb4e2, pol-ef92b8ee4386, vuln-cfad830c2339, vuln-eaee0108862a, vuln-f5d731b5d433
Trace:    router -> remediation -> retriever -> 6 chunks -> specialist[remediation] -> answered
Q: Which of the critical vulnerabilities should I patch first?

Summary:
  Based on the retrieved findings and policy context, prioritize the critical/high items on internet-facing hosts and patch within the policy SLA.

Rem

## 5 · Interactive — ask your own questions

Run this cell and type questions (blank line to stop).

In [8]:
while True:
    q = input("\nAsk the vuln bot (blank to stop): ").strip()
    if not q:
        break
    ask(q)



Ask the vuln bot (blank to stop): how can I remediate Fortinet FortiMail (CVE-2026-104286): 
Q: how can I remediate Fortinet FortiMail (CVE-2026-104286):

Summary:
  Based on the retrieved findings and policy context, prioritize the critical/high items on internet-facing hosts and patch within the policy SLA.

Remediation steps:
  1. Apply the vendor-listed patch or upgrade for the affected plugin.
  2. Re-scan the host to confirm the finding is resolved.
  3. Record the change and closure in the ticketing system.

Prioritization: Ranked by severity and VPR, weighted toward exposed hosts, against the remediation SLA in the retrieved policy.

Citations: pol-bdf56af187ef, pol-ef92b8ee4386, pol-cb8cb70cb4e2, vuln-69f6abff983e, vuln-7c229b146011, vuln-d913bba915d2
Trace:    router -> remediation -> retriever -> 6 chunks -> specialist[remediation] -> answered

Ask the vuln bot (blank to stop): 


## 6 · Simulate an Azure Defender / MDVM scan

This is the Azure path: instead of a Tenable CSV, the `FakeDefenderConnector` returns the bundled sample findings for **2 servers + 2 Windows 11 PCs** (the same shape the live Defender for Endpoint API returns). They're mapped into the bot's `Vulnerability` schema, ingested, and become answerable — exactly as a real scan would.

(To point at a *real* Azure tenant instead, set `AZURE_TENANT_ID` / `AZURE_CLIENT_ID` / `AZURE_CLIENT_SECRET`, `pip install requests`, and swap `FakeDefenderConnector` for `DefenderConnector` — see the project README.)

In [10]:
from vulnrag.connectors import FakeDefenderConnector
from vulnrag.connectors.defender import findings_to_vulnerabilities
from vulnrag.export import build_findings_document

devices = ["srv-web-01", "srv-sql-01", "pc-hr-07", "pc-eng-12"]
conn = FakeDefenderConnector.from_json_file("data/sample_defender_findings.json")
vulns = findings_to_vulnerabilities(conn.get_findings(devices))
store.upsert_vulnerabilities(vulns)   # now queryable by the same bot

doc = build_findings_document(vulns)
print("Simulated Defender scan of", len({v.host for v in vulns}), "devices")
print("  findings :", doc["summary"]["total_findings"])
print("  severity :", doc["summary"]["severity_counts"])
print()
ask("What are the critical findings on srv-web-01 and how do I remediate them?")


Simulated Defender scan of 4 devices
  findings : 13
  severity : {'critical': 3, 'high': 6, 'medium': 3, 'low': 1, 'info': 0}

Q: What are the critical findings on srv-web-01 and how do I remediate them?

Summary:
  Based on the retrieved findings and policy context, prioritize the critical/high items on internet-facing hosts and patch within the policy SLA.

Remediation steps:
  1. Apply the vendor-listed patch or upgrade for the affected plugin.
  2. Re-scan the host to confirm the finding is resolved.
  3. Record the change and closure in the ticketing system.

Prioritization: Ranked by severity and VPR, weighted toward exposed hosts, against the remediation SLA in the retrieved policy.

Citations: pol-bdf56af187ef, pol-cb8cb70cb4e2, pol-ef92b8ee4386, vuln-69f6abff983e, vuln-7c229b146011, vuln-cfad830c2339
Trace:    router -> remediation -> retriever -> 6 chunks -> specialist[remediation] -> answered


### 🚀 Consolidated Non-Optional Pipeline (Run this to execute all steps at once)

In [2]:
# 1. Clone & Install Dependencies
!git clone --depth 1 --branch claude/ai-secops-rag https://github.com/Konaguy/Portfolio.git 2>/dev/null || echo "Already cloned"
%cd /content/Portfolio/vuln-management-rag
!pip install -q "anthropic>=0.40" "langgraph>=0.2.28" "qdrant-client>=1.9" "pydantic>=2.5"

# 2. Build Corpus and Agent Graph
import os
from vulnrag.config import Settings
from vulnrag.embeddings import build_embedder
from vulnrag.vectorstore import VectorStore
from vulnrag.ingest import parse_tenable_file, parse_policy_dir
from vulnrag.agents.graph import VulnRagGraph
from vulnrag.llm import ClaudeClient, FakeLLMClient

settings = Settings.from_env()
store = VectorStore(settings, build_embedder(settings))
n_v = store.upsert_vulnerabilities(parse_tenable_file("data/sample_tenable_export.csv"))
n_p = store.upsert_policies(parse_policy_dir("data/sample_policies"))

# Fallback to FakeLLM since no key is supplied in non-optional flow
llm = FakeLLMClient()
graph = VulnRagGraph.from_settings(settings, llm, store=store)
print(f"Ingested {n_v} vulnerability findings + {n_p} policy chunks -> {store.count()} corpus chunks\n")

# 3. Simulate Azure Defender / MDVM scan
from vulnrag.connectors import FakeDefenderConnector
from vulnrag.connectors.defender import findings_to_vulnerabilities
from vulnrag.export import build_findings_document

devices = ["srv-web-01", "srv-sql-01", "pc-hr-07", "pc-eng-12"]
conn = FakeDefenderConnector.from_json_file("data/sample_defender_findings.json")
vulns = findings_to_vulnerabilities(conn.get_findings(devices))
store.upsert_vulnerabilities(vulns)

doc = build_findings_document(vulns)
print("Simulated Defender scan of", len({v.host for v in vulns}), "devices")
print("  findings :", doc["summary"]["total_findings"])
print("  severity :", doc["summary"]["severity_counts"])
print("\nRunning core queries:\n" + "="*40)

# 4. Run Core Queries
def ask(q):
    final = graph.answer(q)
    a = final["answer"]
    print("Q:", q)
    print("\nSummary:\n ", a["summary"])
    if a["remediation_steps"]:
        print("\nRemediation steps:")
        for i, s in enumerate(a["remediation_steps"], 1):
            print(f"  {i}. {s}")
    print("\nCitations:", ", ".join(a["cited_chunk_ids"]) or "(none)")
    print("Trace:   ", " -> ".join(final.get("trace", [])))
    print("=" * 78)

ask("How do I remediate the Log4Shell findings on the DMZ hosts?")
ask("What are the critical findings on srv-web-01 and how do I remediate them?")

/content/Portfolio/vuln-management-rag
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 25.1 MB/s eta 0:00:00
Ingested 10 vulnerability findings + 9 policy chunks -> 19 corpus chunks

Simulated Defender scan of 4 devices
  findings : 13
  severity : {'critical': 3, 'high': 6, 'medium': 3, 'low': 1, 'info': 0}

Running core queries:
Q: How do I remediate the Log4Shell findings on the DMZ hosts?

Summary:
  Based on the retrieved findings and policy context, prioritize the critical/high items on internet-facing hosts and patch within the policy SLA.

Remediation steps:
  1. Apply the vendor-listed patch or upgrade for the affected plugin.
  2. Re-scan the host to confirm the finding is resolved.
  3. Record the change and closure in the ticketing system.

Citations: pol-bdf56af187ef, pol-cb8cb70cb4e2, pol-ef92b8ee4386, vuln-cfad830c2339, vuln-eaee0108862a, vuln-f5d731b5d433
Trace:    router -> remediat

## 7 · (Optional, advanced) Launch the real Chainlit chat UI

The cells above are the reliable simulation. If you want the actual chat UI, this starts the Chainlit app and exposes it with a Cloudflare quick tunnel (no account needed). Tunnels in Colab are best-effort — if it doesn't come up, stick with the headless cells above.

In [ ]:
import subprocess, time, re, threading, os
from google.colab import output
from google.colab.output import eval_js

# Ensure we are in the correct directory where app.py lives
os.chdir('/content/Portfolio/vuln-management-rag')

# Install Chainlit ignoring the google-adk version constraints since they don't impact this deployment
!pip install -q chainlit --use-deprecated=legacy-resolver

# Run Chainlit app headlessly on port 8000
subprocess.Popen(["chainlit", "run", "app.py", "--host", "0.0.0.0", "--port", "8000", "--headless"])
time.sleep(8)

# Generate the proxy URL using Colab's port forwarder
proxy_url = eval_js("google.colab.kernel.proxyPort(8000)")

# Provide a direct clickable link to open the GUI in a new tab
from IPython.display import display, HTML
display(HTML(f'''
<div style="padding: 15px; background-color: #e6f4ea; border: 1px solid #34a853; border-radius: 8px; margin: 10px 0;">
    <h3 style="color: #137333; margin-top: 0;">🚀 Web GUI Ready!</h3>
    <p>Click the button below to open your Vulnerability Management bot in a new page:</p>
    <a href="{proxy_url}" target="_blank" style="
        display: inline-block;
        padding: 10px 20px;
        background-color: #1a73e8;
        color: white;
        text-decoration: none;
        border-radius: 4px;
        font-weight: bold;
    ">Open Web GUI in New Tab</a>
</div>
'''))